In [ ]:
# Run from the repo root so `agents` and `policy_docs/` resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [3]:
%pip install "snowflake-connector-python[pandas]" pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import snowflake.connector
from getpass import getpass

conn = snowflake.connector.connect(
    account="VGIFDCN-FU80974",
    user="tejastadla",
    password=getpass("Snowflake password: "),
    warehouse="COMPUTE_WH",
    database="TRANSITWISE_DB",
    schema="RAW",
)

conn.cursor().execute("SELECT * FROM ROUTES LIMIT 5").fetch_pandas_all()

,ROUTE_ID,LONG_NAME,SHORT_NAME,ROUTE_TYPE,DESCRIPTION,COLOR,DIRECTION_NAMES,LOADED_AT
0,Red,Red Line,,1,Rapid Transit,DA291C,"South, North",2026-10-08 15:52:12.870438
1,Mattapan,Mattapan Line,,0,Rapid Transit,DA291C,"Outbound, Inbound",2026-10-08 15:52:12.870438
2,Orange,Orange Line,,1,Rapid Transit,ED8B00,"South, North",2026-10-08 15:52:12.870438
3,Green-B,Green Line B,B,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438
4,Green-C,Green Line C,C,0,Rapid Transit,00843D,"West, East",2026-10-08 15:52:12.870438


In [5]:
cur = conn.cursor()
for t in ["ROUTES", "STATIONS", "ROUTE_STOPS", "FACILITIES", "ALERTS_HISTORY"]:
    print(t, cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows")

ROUTES 21 rows
STATIONS 257 rows
ROUTE_STOPS 337 rows
FACILITIES 584 rows
ALERTS_HISTORY 513 rows


In [6]:
cur.execute("SELECT * FROM FACILITIES LIMIT 10").fetch_pandas_all()


,FACILITY_ID,TYPE,SHORT_NAME,LONG_NAME,STATION_ID,LOADED_AT
0,947,ELEVATOR,Sydney Street to unpaid lobby,Savin Hill Elevator 947 (Sydney Street to unpa...,place-shmnl,2026-10-08 17:08:32.485283
1,portlift-brmnl-1,PORTABLE_BOARDING_LIFT,Mobile lift (Park Street & North),Brigham Circle mobile lift (Park Street & North),place-brmnl,2026-10-08 17:08:32.485283
2,821,ELEVATOR,Lobby to Harvard Square,Harvard Elevator 821 (Lobby to Harvard Square),place-harsq,2026-10-08 17:08:32.485283
3,subplat-GB-0296-1,ELEVATED_SUBPLATFORM,Mini-high platform (Track 1/Outbound),West Gloucester mini-high platform (Track 1/Ou...,place-GB-0296,2026-10-08 17:08:32.485283
4,subplat-NHRML-0218-2,ELEVATED_SUBPLATFORM,Mini-high platform,North Billerica mini-high platform,place-NHRML-0218,2026-10-08 17:08:32.485283
5,147,ESCALATOR,Forest Hills platform to Washington Street lobby,Tufts Medical Center Escalator 147 (Forest Hil...,place-tumnl,2026-10-08 17:08:32.485283
6,765,ELEVATOR,Platform to Medford Street lobby,Gilman Square Elevator 765 (Platform to Medfor...,place-gilmn,2026-10-08 17:08:32.485283
7,510,ESCALATOR,Ashmont/Braintree platform to paid lobby,Porter Escalator 510 (Ashmont/Braintree platfo...,place-portr,2026-10-08 17:08:32.485283
8,997,ELEVATOR,Bridge St to parking garage to busway and plat...,Salem Elevator 997 (Bridge St to parking garag...,place-ER-0168,2026-10-08 17:08:32.485283
9,516,ESCALATOR,Alewife platform to Church Street lobby,Harvard Escalator 516 (Alewife platform to Chu...,place-harsq,2026-10-08 17:08:32.485283


In [7]:
import requests

from getpass import getpass
MBTA_KEY = getpass("MBTA key: ")

r = requests.get(
    "https://api-v3.mbta.com/alerts",
    params={"filter[activity]": "USING_WHEELCHAIR", "filter[datetime]": "NOW"},
    headers={"x-api-key": MBTA_KEY},
)
for a in r.json()["data"][:5]:
    print("-", a["attributes"]["header"])

- Natick Center Elevator 751 (Track 2 to street) is unavailable.
- The Central Avenue mini-high platform (Ashmont side) is out of service. 
- On Tuesday, September 22, through Tuesday, December 1, State Elevator 803 (Forest Hills platform to Washington Street, Milk Street) will be unavailable due to maintenance.
- Worcester Elevator 782 (Parking lot to pedestrian bridge) is unavailable. Please use elevator in main building at 2 Washington Sq.
- On Tuesday, September 22, through Tuesday, December 1, Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) will be unavailable due to maintenance.


In [ ]:
cur = conn.cursor()
for t in ["STATIONS", "ROUTE_STOPS", "FACILITIES", "ALERTS_HISTORY"]:
    print(t)
    print(cur.execute(f"SELECT * FROM {t} LIMIT 3").fetch_pandas_all().to_string())
    print()

STATIONS
       STATION_ID           NAME MUNICIPALITY                                          ADDRESS   LATITUDE  LONGITUDE  WHEELCHAIR_BOARDING WHEELCHAIR_STATUS                  LOADED_AT
0     place-sstat  South Station       Boston               700 Atlantic Ave, Boston, MA 02110  42.352271 -71.055242                    1        accessible 2026-10-08 16:06:49.434171
1  place-WML-0199    West Natick       Natick            249 West Central St, Natick, MA 01760  42.283064 -71.391797                    1        accessible 2026-10-08 16:06:49.434171
2     place-cedgr    Cedar Grove       Boston  Fellsway St and Milton St, Dorchester, MA 02124  42.279629 -71.060394                    1        accessible 2026-10-08 16:06:49.434171

ROUTE_STOPS
  ROUTE_ID  STOP_SEQUENCE   STATION_ID STATION_NAME                  LOADED_AT
0      Red              1  place-alfcl      Alewife 2026-10-08 16:16:57.727707
1      Red              2  place-davis        Davis 2026-10-08 16:16:57.727707
2      Re

: 